# 1. Data stage (CPU session, no GPU quota used)

Kaggle settings (right panel): **Accelerator: None**, **Internet: On**.
Add your Hugging Face token under **Add-ons -> Secrets** with the name `HF_TOKEN`
(only English/Svarah needs it).

Run with **Save Version -> Save & Run All (Commit)** so it keeps running with the
browser closed. When it finishes, the notebook's Output (`/kaggle/working`) holds the
prepared features; the training notebook attaches it as an input.

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/snehithareddy28/fastspeech2-assignment.git"
os.chdir("/kaggle/working")
if not os.path.exists("fastspeech2-assignment"):
    subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir("/kaggle/working/fastspeech2-assignment")
!git pull
!pip install -q -r requirements.txt

In [ ]:
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("No HF_TOKEN secret (needed for English):", e)

In [ ]:
!bash scripts/setup_mfa.sh

In [ ]:
!bash scripts/run_pipeline.sh data telugu 2>&1 | tail -60

In [ ]:
!bash scripts/run_pipeline.sh data english 2>&1 | tail -60

In [ ]:
# Summaries to sanity-check before training
!cat Results/Telugu/data_selection_summary.json Results/Telugu/features_summary.json Results/Telugu/alignment_coverage.txt
!cat Results/English/data_selection_summary.json Results/English/features_summary.json Results/English/alignment_coverage.txt

In [ ]:
# Keep only what training needs in the output (MFA env and raw wavs are large)
!rm -rf .mfa_env .micromamba mfa_models data/*/raw/*/ data/*/TextGrid
!du -sh data/*/preprocessed